In [1]:
import tkinter as tk
import numpy as np
import cv2
import joblib
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf

# ============================
# 1. Antrenare model KNN pe MNIST
# ============================

# Încarc setul de date MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Preprocesare: flatten și normalizare pixeli la [0,1]
x_train = x_train.reshape(-1, 28*28) / 255.0
x_test = x_test.reshape(-1, 28*28) / 255.0

# Împărțim setul de antrenament pentru validare rapidă
X_train, X_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.1, random_state=42)

# Definim și antrenăm modelul KNN cu ponderi în funcție de distanță
knn = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn.fit(X_train, y_train)

# Evaluăm pe setul de validare
y_val_pred = knn.predict(X_val)
val_acc = np.mean(y_val_pred == y_val)
print(f"Acuratețe validare: {val_acc:.4f}")

# Salvăm modelul antrenat
joblib.dump(knn, 'knn_mnist_model2.pkl')

Acuratețe validare: 0.9747


['knn_mnist_model2.pkl']

In [2]:
# ============================
# 2. Aplicație Tkinter pentru desen și predicție
# ============================

class DrawingApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Recunoaștere cifre KNN")

        # Canvas 280x280 pentru desen
        self.canvas = tk.Canvas(root, width=280, height=280, bg='white')
        self.canvas.pack()
        self.canvas.bind('<B1-Motion>', self.paint)

        # Butoane
        btn_frame = tk.Frame(root)
        btn_frame.pack(pady=5)
        tk.Button(btn_frame, text='Recunoaște', command=self.recognize).pack(side='left', padx=5)
        tk.Button(btn_frame, text='Șterge', command=self.clear).pack(side='left', padx=5)

        # Etichete rezultat
        self.result_label = tk.Label(root, text='Cifra prezisă: -', font=('Arial', 14))
        self.result_label.pack()

        # Imagine internă 280x280
        self.image = np.zeros((280,280), dtype=np.uint8)
        self.drawing = False

        # Încarc modelul KNN salvat
        self.model = joblib.load('knn_mnist_model2.pkl')

    def paint(self, event):
        x, y = event.x, event.y
        if 0 <= x < 280 and 0 <= y < 280:
            # desenăm un cerc pentru grosime
            self.canvas.create_oval(x-8, y-8, x+8, y+8, fill='black', outline='black')
            cv2.circle(self.image, (x,y), 8, 255, -1)
            self.drawing = True

    def clear(self):
        self.canvas.delete('all')
        self.image.fill(0)
        self.result_label.config(text='Cifra prezisă: -')
        self.drawing = False

    def recognize(self):
        if not self.drawing:
            return
        # Preprocesare imagine desenată
        img28 = cv2.resize(self.image, (28,28))
        img28 = cv2.bitwise_not(img28)  # invertim: fundal 0, cifră albă
        img28 = img28.reshape(1, -1) / 255.0

        # Predict și afișare
        probs = self.model.predict_proba(img28)[0]
        pred = np.argmax(probs)
        self.result_label.config(text=f'Cifra: {pred}, probabilitate: {probs[pred]:.2f}')
        print('Probabilități:', np.round(probs,2))
        self.drawing = False

# Pornim aplicația
if __name__ == '__main__':
    root = tk.Tk()
    app = DrawingApp(root)
    root.mainloop()


Probabilități: [0.8 0.  0.  0.  0.  0.  0.  0.  0.2 0. ]
Probabilități: [0.8 0.  0.  0.  0.  0.  0.  0.  0.2 0. ]
Probabilități: [0.  0.  0.6 0.  0.  0.  0.  0.  0.4 0. ]
Probabilități: [0.6 0.  0.  0.  0.  0.  0.2 0.  0.2 0. ]
Probabilități: [0.4 0.  0.2 0.  0.  0.  0.  0.  0.4 0. ]
Probabilități: [0.6 0.  0.  0.  0.  0.2 0.  0.  0.2 0. ]
